# Summary Statistics Table

Produces a descriptive-statistics table (mean, SD, P25, median, P75, N) for
the four key variables used in the empirical analysis: employment, wages,
LM_AIOE and MS_SCORE. Two versions are produced:

1. **Occupation-industry-year level** -- the raw panel used in the TWFE analysis.
2. **Occupation level** -- one row per occupation (LM_AIOE and MS_SCORE only
   vary at this level, so their occupation-level distribution is the more
   meaningful one to report).

Intended for the Data Appendix.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('bls_onet_felten_ms_panel_harmonised.csv')
df = df[~df['year'].isin([2020, 2021])].copy()

df['TOT_EMP']  = pd.to_numeric(df['TOT_EMP'], errors='coerce')
df['A_MEDIAN'] = pd.to_numeric(df['A_MEDIAN'], errors='coerce')

print(f'Rows: {len(df)}')


## Occupation-industry-year level summary


In [ ]:
def summarise(series):
    s = series.dropna()
    return pd.Series({
        'N': int(s.shape[0]),
        'Mean': s.mean(),
        'SD': s.std(),
        'P25': s.quantile(0.25),
        'Median': s.median(),
        'P75': s.quantile(0.75),
    })

cell_level_vars = {
    'Employment (TOT_EMP)': df['TOT_EMP'],
    'Annual median wage, USD (A_MEDIAN)': df['A_MEDIAN'],
    'LM_AIOE': df['LM_AIOE'],
    'MS_SCORE': df['MS_SCORE'],
}

cell_level_summary = pd.DataFrame({name: summarise(s) for name, s in cell_level_vars.items()}).T
cell_level_summary = cell_level_summary[['N', 'Mean', 'SD', 'P25', 'Median', 'P75']]
cell_level_summary.to_csv('summary_stats_occupation_industry_year.csv')
print('Occupation-industry-year level (raw panel used in TWFE):')
print(cell_level_summary.round(2).to_string())


## Occupation level summary (one row per occupation)

LM_AIOE and MS_SCORE only vary at the occupation level, so reporting their
distribution at the occupation-industry-year level double-counts the same
value across every industry and year an occupation appears in. This section
collapses to one row per occupation first.


In [ ]:
occ_level = df.drop_duplicates(subset='OCC_CODE')[['OCC_CODE', 'LM_AIOE', 'MS_SCORE']]

# Occupation-level employment and wage: summed / employment-weighted across
# industries and years, matching the aggregation used in the SC/GSC panel.
occ_year = (
    df.groupby(['OCC_CODE', 'year'])
    .apply(lambda g: pd.Series({
        'TOT_EMP_occ_year': g['TOT_EMP'].sum(min_count=1),
        'log_wage_occ_year': np.average(
            np.log(g['A_MEDIAN'].replace(0, np.nan).dropna()),
            weights=g.loc[g['A_MEDIAN'].notna(), 'TOT_EMP'].fillna(0) + 1e-9
        ) if g['A_MEDIAN'].notna().any() else np.nan,
    }))
    .reset_index()
)

occ_level_vars = {
    'LM_AIOE (occupation level)': occ_level['LM_AIOE'],
    'MS_SCORE (occupation level)': occ_level['MS_SCORE'],
    'Employment, occupation-year (summed across industries)': occ_year['TOT_EMP_occ_year'],
    'Log wage, occupation-year (employment-weighted)': occ_year['log_wage_occ_year'],
}

occ_level_summary = pd.DataFrame({name: summarise(s) for name, s in occ_level_vars.items()}).T
occ_level_summary = occ_level_summary[['N', 'Mean', 'SD', 'P25', 'Median', 'P75']]
occ_level_summary.to_csv('summary_stats_occupation_level.csv')
print('Occupation level (one row per occupation, or per occupation-year for outcomes):')
print(occ_level_summary.round(3).to_string())


## Correlation and coverage summary

Reproduces the LM_AIOE/MS_SCORE correlation and match-rate figures already
reported in Section 3.1/3.3, alongside the occupation counts, for
convenience when building the appendix table.


In [ ]:
n_occ_total   = df['OCC_CODE'].nunique()
n_occ_aioe    = df.loc[df['LM_AIOE'].notna(), 'OCC_CODE'].nunique()
n_occ_ms      = df.loc[df['MS_SCORE'].notna(), 'OCC_CODE'].nunique()
n_occ_both    = df.loc[df['LM_AIOE'].notna() & df['MS_SCORE'].notna(), 'OCC_CODE'].nunique()
corr          = occ_level[['LM_AIOE', 'MS_SCORE']].corr().iloc[0, 1]

coverage = pd.Series({
    'Total occupations in panel': n_occ_total,
    'Occupations matched to LM_AIOE': n_occ_aioe,
    'Occupations matched to MS_SCORE': n_occ_ms,
    'Occupations matched to both indices': n_occ_both,
    'Correlation, LM_AIOE vs MS_SCORE (occupation level)': round(corr, 3),
})
coverage.to_csv('summary_stats_coverage.csv', header=['value'])
print(coverage.to_string())
